# LifeMate — Zalo AI Challenge 2026
Zero-shot baseline → SFT fine-tune → submission

**Model**: Qwen3-4B (4.02B params) | **GPU**: T4 | **Budget**: 30s/sample

## 1. Setup

In [ ]:
!pip install unsloth transformers accelerate peft bitsandbytes trl datasets -q
!rm -rf /tmp/repo && git clone https://github.com/Justinbiahoi-coding/zalo-lifemate-2026.git /tmp/repo
!cp /tmp/repo/*.py /tmp/repo/*.json .
!ls *.py *.json

In [ ]:
ROOT = "/kaggle/input/datasets/thinonlineinterau1bi/lifemate/train+public_test"

## 2. Generate SFT Training Data

In [ ]:
!python make_sft.py --root {ROOT} --dev split_dev.json --out sft_train.jsonl

## 3. Fine-tune (QLoRA NF4)

In [ ]:
!python train_sft.py --data sft_train.jsonl --epochs 5 --r 64 --out lora_out

## 4. Eval — Fine-tuned vs Zero-shot (dev set, 347 mẫu)

In [ ]:
import os
if os.path.exists("lora_out/adapter_config.json"):
    !python baseline.py --root {ROOT} --split train --dev split_dev.json \
        --lora lora_out --no-think --out sub_ft.jsonl --score
else:
    print("⚠ lora_out not found — training likely failed, skipping fine-tuned eval")

In [ ]:
# Zero-shot comparison
!python baseline.py --root {ROOT} --split train --dev split_dev.json \
    --no-think --out sub_zs.jsonl --score --limit 50